## Step 0 — Load data
Run this cell first. It uses `/content/archive.zip` which you already uploaded to Colab.

In [ ]:
# ── Run this FIRST in every new Colab session ─────────────────────
import zipfile, os, numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
import pickle

with zipfile.ZipFile("/content/archive.zip", "r") as z:
    z.extractall("dataset")
print("Unzipped.")

DATA_ROOT = "dataset/data"
X_list, y_list, g_list = [], [], []

for act_folder in sorted(os.listdir(DATA_ROOT)):
    act_path = os.path.join(DATA_ROOT, act_folder)
    if not os.path.isdir(act_path) or not act_folder.startswith("a"):
        continue
    act_id = int(act_folder[1:]) - 1
    for subj_folder in sorted(os.listdir(act_path)):
        subj_path = os.path.join(act_path, subj_folder)
        if not os.path.isdir(subj_path): continue
        subj_id = int(subj_folder[1:])
        for seg_file in sorted(os.listdir(subj_path)):
            if not seg_file.endswith(".txt"): continue
            fpath = os.path.join(subj_path, seg_file)
            try: data = np.loadtxt(fpath, delimiter=",")
            except: continue
            if data.ndim != 2 or data.shape[1] != 45: continue
            X_list.append(data); y_list.append(act_id); g_list.append(subj_id)

X = np.array(X_list, dtype=np.float32)
y = np.array(y_list, dtype=np.int64)
g = np.array(g_list, dtype=np.int64)
print(f"Loaded: X={X.shape}  activities={len(set(y.tolist()))}  subjects={sorted(set(g.tolist()))}")

sp1 = GroupShuffleSplit(1, test_size=0.15, random_state=42)
tv_idx, te_idx = next(sp1.split(X, y, g))
sp2 = GroupShuffleSplit(1, test_size=0.15/0.85, random_state=42)
tr_idx, va_idx = next(sp2.split(X[tv_idx], y[tv_idx], g[tv_idx]))

X_train, y_train = X[tv_idx][tr_idx], y[tv_idx][tr_idx]
X_val,   y_val   = X[tv_idx][va_idx], y[tv_idx][va_idx]
X_test,  y_test  = X[te_idx],          y[te_idx]

N, T, C = X_train.shape
scaler = StandardScaler()
scaler.fit(X_train.reshape(-1, C))
def tf(X):
    n,t,c = X.shape
    return scaler.transform(X.reshape(-1,c)).reshape(n,t,c).astype(np.float32)
X_train = tf(X_train); X_val = tf(X_val); X_test = tf(X_test)

os.makedirs("data/processed", exist_ok=True)
for name, arr in [("X_train",X_train),("y_train",y_train),
                   ("X_val",X_val),("y_val",y_val),
                   ("X_test",X_test),("y_test",y_test)]:
    np.save(f"data/processed/{name}.npy", arr)
with open("data/processed/scaler.pkl","wb") as f: pickle.dump(scaler, f)
print("Done. Train={} Val={} Test={}".format(len(X_train), len(X_val), len(X_test)))
# ──────────────────────────────────────────────────────────────────


# Week 3 — BiLSTM Model

> Run `week2_preprocessing.ipynb` first.

## Step 0 — Imports

In [ ]:
import os, json, pickle, warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
os.makedirs("results", exist_ok=True)

ACTIVITY_NAMES = [
    "Sitting","Standing","Lying-Back","Lying-Right",
    "Asc-Stairs","Desc-Stairs","Stand-Elev","Move-Elev",
    "Walk-Park","Walk-Flat","Walk-15deg","Run",
    "Stepper","Cross-Trainer","Bike-Horiz",
    "Bike-Vert","Rowing","Jumping","Basketball",
]


## Step 1 — Load data

In [ ]:
splits = {}
for k in ("X_train","y_train","X_val","y_val","X_test","y_test"):
    path = f"data/processed/{k}.npy"
    if not os.path.isfile(path):
        raise FileNotFoundError(
            f"Not found: {path}\n"
            "Run week2_preprocessing.ipynb first."
        )
    splits[k] = np.load(path)
print(f"Train={splits['X_train'].shape}")
print(f"Val  ={splits['X_val'].shape}")
print(f"Test ={splits['X_test'].shape}")


## Step 2 — DataLoaders
LSTM uses (N, time, channels) — no transpose needed.

In [ ]:
def make_loader(X, y, bs=64, shuffle=True):
    Xt = torch.tensor(X, dtype=torch.float32)
    yt = torch.tensor(y, dtype=torch.long)
    return DataLoader(TensorDataset(Xt, yt), batch_size=bs, shuffle=shuffle)

train_loader = make_loader(splits["X_train"], splits["y_train"], shuffle=True)
val_loader   = make_loader(splits["X_val"],   splits["y_val"],   shuffle=False)
test_loader  = make_loader(splits["X_test"],  splits["y_test"],  shuffle=False)
print(f"Train batches={len(train_loader)}  Val={len(val_loader)}  Test={len(test_loader)}")


## Step 3 — BiLSTM model
2-layer bidirectional LSTM. Reads signal forward and backward.

In [ ]:
class BiLSTM(nn.Module):
    def __init__(self, in_size=45, hidden=128, layers=2, num_cls=19, drop=0.3):
        super().__init__()
        self.lstm = nn.LSTM(in_size, hidden, layers, batch_first=True,
                            bidirectional=True,
                            dropout=drop if layers>1 else 0.0)
        self.norm = nn.LayerNorm(hidden*2)
        self.fc   = nn.Sequential(
            nn.Dropout(0.5), nn.Linear(hidden*2, 128), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(128, num_cls),
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(self.norm(out[:,-1,:]))

model = BiLSTM().to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(model)


## Step 4 — Training setup

In [ ]:
EPOCHS, LR, PATIENCE = 50, 1e-3, 10
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, "min", factor=0.5, patience=5)
print(f"Epochs={EPOCHS}  LR={LR}  Patience={PATIENCE}")


## Step 5 — Train

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None, is_lstm=True):
    model.train() if optimizer else model.eval()
    total_loss, correct, total = 0, 0, 0
    ctx = torch.enable_grad() if optimizer else torch.no_grad()
    with ctx:
        for Xb, yb in loader:
            Xb, yb = Xb.to(device), yb.to(device)
            out  = model(Xb)
            loss = criterion(out, yb)
            if optimizer:
                optimizer.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total_loss += loss.item()*len(yb)
            correct    += (out.argmax(1)==yb).sum().item()
            total      += len(yb)
    return total_loss/total, correct/total

history = {"tl":[],"vl":[],"ta":[],"va":[]}
best_vl, patience_cnt, best_w = 1e9, 0, None

print(f"{'Ep':>4} {'TrainLoss':>10} {'TrainAcc':>9} {'ValLoss':>9} {'ValAcc':>8}")
print("-"*46)
for ep in range(1, EPOCHS+1):
    tl,ta = run_epoch(model, train_loader, criterion, optimizer)
    vl,va = run_epoch(model, val_loader,   criterion)
    scheduler.step(vl)
    for k,v in zip(["tl","vl","ta","va"],[tl,vl,ta,va]):
        history[k].append(v)
    mark = ""
    if vl < best_vl:
        best_vl = vl
        best_w  = {k:v.cpu().clone() for k,v in model.state_dict().items()}
        patience_cnt = 0; mark = " *"
    else:
        patience_cnt += 1
    if ep%5==0 or ep==1:
        print(f"{ep:>4} {tl:>10.4f} {ta:>9.4f} {vl:>9.4f} {va:>8.4f}{mark}")
    if patience_cnt >= PATIENCE:
        print(f"Early stop at epoch {ep}"); break

model.load_state_dict(best_w)
print(f"\nBest val loss: {best_vl:.4f}")


## Step 6 — Loss and accuracy curves

In [ ]:
fig, (ax1,ax2) = plt.subplots(1,2,figsize=(12,4))
ep = range(1, len(history["tl"])+1)
ax1.plot(ep, history["tl"], label="Train"); ax1.plot(ep, history["vl"], label="Val")
ax1.set_title("Loss — BiLSTM"); ax1.set_xlabel("Epoch"); ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(ep, history["ta"], label="Train"); ax2.plot(ep, history["va"], label="Val")
ax2.set_title("Accuracy — BiLSTM"); ax2.set_xlabel("Epoch"); ax2.legend(); ax2.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("results/week3_lstm_curves.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: results/week3_lstm_curves.png")


## Step 7 — Test evaluation

In [ ]:
model.eval()
preds, labels = [], []
with torch.no_grad():
    for Xb, yb in test_loader:
        preds.append(model(Xb.to(device)).argmax(1).cpu().numpy())
        labels.append(yb.numpy())
preds  = np.concatenate(preds)
labels = np.concatenate(labels)

lstm_acc = accuracy_score(labels, preds)
lstm_f1  = f1_score(labels, preds, average="macro")
lstm_cm  = confusion_matrix(labels, preds)

print(f"Test accuracy={lstm_acc:.4f}  F1={lstm_f1:.4f}")
print()
print(classification_report(labels, preds, target_names=ACTIVITY_NAMES, digits=3))


## Step 8 — Confusion matrix

In [ ]:
cm_pct = lstm_cm.astype(float)/lstm_cm.sum(axis=1,keepdims=True)*100
fig, ax = plt.subplots(figsize=(13,10))
sns.heatmap(cm_pct, annot=True, fmt=".0f", cmap="Purples",
            xticklabels=ACTIVITY_NAMES, yticklabels=ACTIVITY_NAMES,
            vmin=0, vmax=100, ax=ax, annot_kws={"size":7}, linewidths=0.3)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Confusion Matrix (%) — BiLSTM", fontweight="bold")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)
fig.tight_layout()
fig.savefig("results/week3_lstm_confusion_matrix.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: results/week3_lstm_confusion_matrix.png")


## Step 9 — Save

In [ ]:
torch.save(model.state_dict(), "results/lstm_model.pth")
np.save("results/lstm_cm.npy", lstm_cm)
with open("results/lstm_history.pkl","wb") as f: pickle.dump(history, f)
metrics = {"test_accuracy":round(float(lstm_acc),4),
           "test_f1":      round(float(lstm_f1),4),
           "epochs":       len(history["tl"])}
with open("results/lstm_metrics.json","w") as f: json.dump(metrics, f, indent=2)
print("Saved.")
print(metrics)
